<a href="https://colab.research.google.com/github/andandandand/practical-computer-vision/blob/main/notebooks/Food_Dataset_Curation_with_Fiftyone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Tutorial: Curating a HuggingFace Image Dataset with FiftyOne

#### Author: [Antonio Rueda-Toicen](antonio@getfiftyone.com)


[![Creative Commons License](https://i.creativecommons.org/l/by/4.0/88x31.png)](http://creativecommons.org/licenses/by/4.0/)

This work is licensed under a [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).

This notebook demonstrates how to curate a dataset for computer vision tasks using FiftyOne and HuggingFace. We will start by loading a dataset from HuggingFace, translate German feature names to English, and then move the dataset into FiftyOne for further processing and analysis.

We apply the [YOLO-E](https://github.com/THU-MIG/yoloe) algorithm to the dataset to produce segmentations and the [DINOv2](https://arxiv.org/abs/2304.07193) network to produce image embeddings.

### Source data


We work with the Food Waste dataset from the AI Service Center at HPI published at HuggingFace. This dataset was compiled by [L. Stroetmann](https://www.stroetmann.de/) and [a la QUARTO](https://ala.quarto.de/).

* https://huggingface.co/datasets/AI-ServicesBB/food-waste-dataset


### Workflow

After translating the feature names and ingredients from German to English and create a FiftyOne dataset with segmentations and image embeddings to help navigate it.

![](https://github.com/andandandand/practical-computer-vision/blob/main/images/food_waste_navigation.webp?raw=1)

In [ ]:
!pip install datasets==3.6.0 fiftyone==1.7.0 > /dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-runtime-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-runtime-cu12 12.5.82 w

In [ ]:
!pip install ultralytics==8.3.164 > /dev/null

In [ ]:
!pip install xformers==0.0.31 > /dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.7.19 requires torch<2.7,>=1.10, but you have torch 2.7.1 which is incompatible.
torchvision 0.21.0+cu124 requires torch==2.6.0, but you have torch 2.7.1 which is incompatible.
torchaudio 2.6.0+cu124 requires torch==2.6.0, but you have torch 2.7.1 which is incompatible.


# Food Waste Dataset

This dataset contains detailed information about food waste, including images and nutritional information.

## Dataset Description

The dataset combines image data with detailed nutritional information for each meal and its ingredients.

### Features

For each entry:
- `bonid`: Unique identifier for each meal
- `image`: Image of the meal
- Lists per meal (multiple ingredients):
  - `Artikelnummer`: Article numbers
  - `Artikel`: Ingredient names
  - `Stückartikel`: Piece article information
  - `Anzahl_Kellen`: Number of portions
  - `Gewicht_Kelle`: Weight per portion
  - `Gewicht_Teller`: Weight per plate
  - `kcal_Teller`, `kj_Teller`: Caloric information
  - `Fett_Teller`, `ges_Fettsäuren_Teller`: Fat content
  - `Kohlenhydrate_Teller`, `Zucker_Teller`: Carbohydrate content
  - `Eiweiß_Teller`: Protein content
  - `Salz_Teller`: Salt content
  - `Menge_Rückläufer`, `Prozent_Rückläufer`: Return quantities

Per image measurements:
- Before consumption:
  - `Gewicht_vorher`: Initial weight
  - `kcal_vorher`, `kj_vorher`: Initial calories
  - `Fett_vorher`, `ges_Fettsäuren_vorher`: Initial fat content
  - `Kohlenhydrate_vorher`, `Zucker_vorher`: Initial carbohydrates
  - `Eiweiß_vorher`: Initial protein
  - `Salz_vorher`: Initial salt
- After consumption:
  - `Gewicht_nachher`: Remaining weight
  - `kcal_nachher`, `kj_nachher`: Remaining calories
  - `Fett_nachher`, `ges_fettsäuren_nachher`: Remaining fat
  - `Kohlenhydrate_nachher`, `Zucker_nachher`: Remaining carbohydrates
  - `Eiweiß_nachher`: Remaining protein
  - `Salz_nachher`: Remaining salt


In [ ]:
from datasets import load_dataset

hf_dataset = load_dataset("AI-ServicesBB/food-waste-dataset")

README.md:   0%|          | 0.00/4.66k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/51.1M [00:00<?, ?B/s]

(…)-00000-of-00001-64762f3000d1dde2.parquet:   0%|          | 0.00/76.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/215 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/160 [00:00<?, ? examples/s]

In [ ]:
# Quick inspection on the dataset
hf_dataset

DatasetDict({
    train: Dataset({
        features: ['bonid', 'image', 'Bon_ID', 'Artikelnummer', 'Artikel', 'Stückartikel', 'Anzahl_Kellen', 'Gewicht_Kelle', 'Gewicht_Teller', 'kcal_Teller', 'kj_Teller', 'Fett_Teller', 'ges_Fettsäuren_Teller', 'Kohlenhydrate_Teller', 'Zucker_Teller', 'Eiweiß_Teller', 'Salz_Teller', 'Menge_Rückläufer', 'Prozent_Rückläufer', 'Gericht', 'Portionsgröße', 'Gewicht_vorher', 'kcal_vorher', 'kj_vorher', 'Fett_vorher', 'ges_Fettsäuren_vorher', 'Kohlenhydrate_vorher', 'Zucker_vorher', 'Eiweiß_vorher', 'Salz_vorher', 'Gewicht_nachher', 'kcal_nachher', 'kj_nachher', 'Fett_nachher', 'ges_fettsäuren_nachher', 'Kohlenhydrate_nachher', 'Zucker_nachher', 'Eiweiß_nachher', 'Salz_nachher'],
        num_rows: 215
    })
    test: Dataset({
        features: ['bonid', 'image', 'Bon_ID', 'Artikelnummer', 'Artikel', 'Stückartikel', 'Anzahl_Kellen', 'Gewicht_Kelle', 'Gewicht_Teller', 'kcal_Teller', 'kj_Teller', 'Fett_Teller', 'ges_Fettsäuren_Teller', 'Kohlenhydrate_Teller',

## Translate German feature names to English

Let's make the dataset a bit more user-friendly to non-German speakers.


In [ ]:
feature_mapping = {
    'bonid': 'bonid',
    'image': 'image',
    'Bon_ID': 'bon_id',
    'Artikelnummer': 'article_number',
    'Artikel': 'ingredient_name',
    'Stückartikel': 'piece_article',
    'Anzahl_Kellen': 'number_of_portions',
    'Gewicht_Kelle': 'weight_per_portion',
    'Gewicht_Teller': 'weight_per_plate',
    'kcal_Teller': 'kcal_per_plate',
    'kj_Teller': 'kj_per_plate',
    'Fett_Teller': 'fat_per_plate',
    'ges_Fettsäuren_Teller': 'saturated_fat_per_plate',
    'Kohlenhydrate_Teller': 'carbohydrates_per_plate',
    'Zucker_Teller': 'sugar_per_plate',
    'Eiweiß_Teller': 'protein_per_plate',
    'Salz_Teller': 'salt_per_plate',
    'Menge_Rückläufer': 'return_quantity',
    'Prozent_Rückläufer': 'return_percentage',
    'Gericht': 'dish',
    'Portionsgröße': 'portion_size',
    'Gewicht_vorher': 'weight_before',
    'kcal_vorher': 'kcal_before',
    'kj_vorher': 'kj_before',
    'Fett_vorher': 'fat_before',
    'ges_Fettsäuren_vorher': 'saturated_fat_before',
    'Kohlenhydrate_vorher': 'carbohydrates_before',
    'Zucker_vorher': 'sugar_before',
    'Eiweiß_vorher': 'protein_before',
    'Salz_vorher': 'salt_before',
    'Gewicht_nachher': 'weight_after',
    'kcal_nachher': 'kcal_after',
    'kj_nachher': 'kj_after',
    'Fett_nachher': 'fat_after',
    'ges_fettsäuren_nachher': 'saturated_fat_after',
    'Kohlenhydrate_nachher': 'carbohydrates_after',
    'Zucker_nachher': 'sugar_after',
    'Eiweiß_nachher': 'protein_after',
    'Salz_nachher': 'salt_after'
}

# Rename the columns in the dataset
hf_dataset = hf_dataset.rename_columns(feature_mapping)



In [ ]:
# Display the updated dataset structure to confirm the renaming
print(hf_dataset)

DatasetDict({
    train: Dataset({
        features: ['bonid', 'image', 'bon_id', 'article_number', 'ingredient_name', 'piece_article', 'number_of_portions', 'weight_per_portion', 'weight_per_plate', 'kcal_per_plate', 'kj_per_plate', 'fat_per_plate', 'saturated_fat_per_plate', 'carbohydrates_per_plate', 'sugar_per_plate', 'protein_per_plate', 'salt_per_plate', 'return_quantity', 'return_percentage', 'dish', 'portion_size', 'weight_before', 'kcal_before', 'kj_before', 'fat_before', 'saturated_fat_before', 'carbohydrates_before', 'sugar_before', 'protein_before', 'salt_before', 'weight_after', 'kcal_after', 'kj_after', 'fat_after', 'saturated_fat_after', 'carbohydrates_after', 'sugar_after', 'protein_after', 'salt_after'],
        num_rows: 215
    })
    test: Dataset({
        features: ['bonid', 'image', 'bon_id', 'article_number', 'ingredient_name', 'piece_article', 'number_of_portions', 'weight_per_portion', 'weight_per_plate', 'kcal_per_plate', 'kj_per_plate', 'fat_per_plate', 'sat

## Translate ingredient names to English

Do you know what ***Hähnchenstreifen*** are? How about ***Schweinenackenbraten***? 🇩🇪

In [ ]:
german_to_english_ingredients_hyphenated = {
    'Fleischbällchen gebrüht': 'poached-meatballs',
    'Reis': 'rice',
    'Paniertes Fischfilet': 'breaded-fish-fillet',
    'Linseneintopf': 'lentil-stew',
    'Apfelmus': 'applesauce',
    'Helle Sauce': 'light-sauce-or-white-sauce',
    'Kartoffelpüree': 'mashed-potatoes',
    'Rinderbraten': 'roast-beef',
    'Semmelknödel': 'bread-dumplings',
    'Grüne Bohnen': 'green-beans',
    'Möhre': 'carrot',
    'Pflanzencreme': 'vegetable-based-cream',
    'Schinken Mettwurst': 'ham-sausage',
    'Paprika': 'paprika-or-bell-pepper',
    'Seelachs': 'pollock-or-coalfish',
    'Bratenjus': 'gravy',
    'Hähnchenstreifen': 'chicken-strips',
    'Eisbergsalat': 'iceberg-lettuce',
    'Rotkohl': 'red-cabbage',
    'Sauerkraut': 'sauerkraut',
    'Reibekuchen': 'potato-pancakes-or-potato-fritters',
    'Krautsalat': 'coleslaw',
    'Schnitzel': 'schnitzel-or-cutlet',
    'Blumenkohl': 'cauliflower',
    'Rostbratwurst': 'grilled-sausage',
    'Braune Sauce': 'brown-sauce',
    'Kartoffeln': 'potatoes',
    'Kartoffelwürfel': 'diced-potatoes',
    'Sahne': 'cream',
    'Zucchini': 'zucchini-or-courgette',
    'Eierspätzle': 'egg-spaetzle)',
    'Pilze': 'mushrooms',
    'Erbsen': 'peas',
    'Wirsing': 'savoy-cabbage',
    'Malzbier-Senf-Sauce': 'malt-beer-mustard-sauce',
    'Dressing Portion': 'dressing-portion',
    'Linsen': 'lentils',
    'Zwiebel': 'onion',
    'Schweinenackenbraten': 'pork-neck-roast',
    'Hähnchen': 'chicken',
    'Tomaten-Curry-Sauce': 'tomato-curry-sauce'
}

In [ ]:
def map_ingredients_to_english(ingredient_list, translation_dict):
    """Maps German ingredient names to English using a provided dictionary."""
    mapped_list = []
    for ingredient in ingredient_list:
        # Strip whitespace before looking up in the dictionary
        stripped_ingredient = ingredient.strip()
        # Use get() with a default value to handle cases where the ingredient is not in the dictionary
        mapped_list.append(translation_dict.get(stripped_ingredient, stripped_ingredient))
    return mapped_list

# Apply the mapping function to the 'ingredient_name' column in both splits
hf_dataset['train'] = hf_dataset['train'].map(
    lambda example: {'ingredient_name': map_ingredients_to_english(example['ingredient_name'], german_to_english_ingredients_hyphenated)}
)

hf_dataset['test'] = hf_dataset['test'].map(
    lambda example: {'ingredient_name': map_ingredients_to_english(example['ingredient_name'], german_to_english_ingredients_hyphenated)}
)

print("Ingredient names mapped to English in both train and test splits using the provided dictionary.")

Map:   0%|          | 0/215 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Ingredient names mapped to English in both train and test splits using the provided dictionary.


## Move the dataset to FiftyOne

Moving a dataset to FiftyOne allows for visualization, exploration, and curation of data. This capability helps in understanding the dataset content, finding errors, and selecting data subsets. The integration with machine learning tools facilitates model evaluation and iteration.

In [ ]:
import fiftyone as fo
import os


# Create a persistent directory for images
persistent_dir = "fiftyone_images"
os.makedirs(persistent_dir, exist_ok=True)

# Create FiftyOne dataset
fiftyone_dataset = fo.Dataset(name="food_waste_dataset")

sample_count = 0
for split in ['train', 'test']:
    split_count = 0
    for item in hf_dataset[split]:
        # Save image to persistent directory with unique filename
        image_filename = f"{split}_{split_count:06d}.jpg"
        image_path = os.path.join(persistent_dir, image_filename)

        # Save the PIL Image to the persistent file
        item['image'].save(image_path)

        # Create FiftyOne sample
        sample = fo.Sample(filepath=image_path)
        sample['split'] = split

        # Add any additional metadata from the original dataset
        for key, value in item.items():
            if key != 'image':  # Skip the image field since we've handled it
                sample[key] = value

        fiftyone_dataset.add_sample(sample)
        split_count += 1
        sample_count += 1

print(f"FiftyOne dataset created with {len(fiftyone_dataset)} samples.")
print(f"Images saved to: {os.path.abspath(persistent_dir)}")



FiftyOne dataset created with 375 samples.
Images saved to: /content/fiftyone_images


## Compute metadata

Get image size (both in dimensions and bytes) and format. These are often useful stats for our dataset.


In [ ]:
fiftyone_dataset.compute_metadata()

Computing metadata...


INFO:fiftyone.core.metadata:Computing metadata...


 100% |█████████████████| 375/375 [128.1ms elapsed, 0s remaining, 2.9K samples/s]     


INFO:eta.core.utils: 100% |█████████████████| 375/375 [128.1ms elapsed, 0s remaining, 2.9K samples/s]     


## Compute segmentations and bounding boxes with YOLO-E

We can prompt [YOLO-E](https://github.com/THU-MIG/yoloe) to produce segmentation of the ingredients on the plate.

In [ ]:
labels = list(german_to_english_ingredients_hyphenated.values())
labels

['poached-meatballs',
 'rice',
 'breaded-fish-fillet',
 'lentil-stew',
 'applesauce',
 'light-sauce-or-white-sauce',
 'mashed-potatoes',
 'roast-beef',
 'bread-dumplings',
 'green-beans',
 'carrot',
 'vegetable-based-cream',
 'ham-sausage',
 'paprika-or-bell-pepper',
 'pollock-or-coalfish',
 'gravy',
 'chicken-strips',
 'iceberg-lettuce',
 'red-cabbage',
 'sauerkraut',
 'potato-pancakes-or-potato-fritters',
 'coleslaw',
 'schnitzel-or-cutlet',
 'cauliflower',
 'grilled-sausage',
 'brown-sauce',
 'potatoes',
 'diced-potatoes',
 'cream',
 'zucchini-or-courgette',
 'egg-spaetzle)',
 'mushrooms',
 'peas',
 'savoy-cabbage',
 'malt-beer-mustard-sauce',
 'dressing-portion',
 'lentils',
 'onion',
 'pork-neck-roast',
 'chicken',
 'tomato-curry-sauce']

## Use ultralytics's YOLO-E


Ultralytics YOLO-E detects objects and provides segmentation masks. This model offers capabilities for locating instances of items within images.

In [ ]:
from ultralytics import YOLOE

segmentation_model = YOLOE("yoloe-11s-seg.pt")

labels = list(german_to_english_ingredients_hyphenated.values())
segmentation_model.set_classes(labels, segmentation_model.get_text_pe(labels))

fiftyone_dataset.apply_model(segmentation_model, label_field="yoloe_segmentation")


WARNING ⚠️ torchvision==0.21 is incompatible with torch==2.7.
Run 'pip install torchvision==0.22' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 26.5M/26.5M [00:00<00:00, 219MB/s]


requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...

requirements: AutoUpdate success ✅ 3.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|██████████| 572M/572M [00:17<00:00, 34.6MB/s]


 100% |█████████████████| 375/375 [20.4s elapsed, 0s remaining, 22.7 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 375/375 [20.4s elapsed, 0s remaining, 22.7 samples/s]      


## Compute patch embeddings

We create embeddings for each of the patches that we have found through our segmnentation masks, this allows us to localize the similarity search on the detected objects.

In [ ]:
import fiftyone.zoo as foz
embedding_model = foz.load_zoo_model("dinov2-vitl14-reg-torch")

# Compute embeddings from your YOLO-E segmented regions
embeddings = fiftyone_dataset.compute_patch_embeddings(
    model=embedding_model,
    patches_field="yoloe_segmentation",  # Your polyline field
    embeddings_field="segment_embeddings",  # Where to store embeddings
    alpha=0.05,  # Slightly expand the polygon boundary by 5%
    batch_size=32,  # Process multiple patches at once
    num_workers=4   # Parallel processing
)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_reg4_pretrain.pth


100%|██████████| 1.13G/1.13G [00:04<00:00, 275MB/s]


Model does not support batching


 100% |█████████████████| 375/375 [2.6m elapsed, 0s remaining, 7.8 samples/s]       


INFO:eta.core.utils: 100% |█████████████████| 375/375 [2.6m elapsed, 0s remaining, 7.8 samples/s]       


In [ ]:
a_sample = fiftyone_dataset.first()
a_sample.get_field_schema()

OrderedDict([('id', <fiftyone.core.fields.ObjectIdField at 0x7ff46913c910>),
             ('filepath',
              <fiftyone.core.fields.StringField at 0x7ff46b368990>),
             ('tags', <fiftyone.core.fields.ListField at 0x7ff46b384810>),
             ('metadata',
              <fiftyone.core.fields.EmbeddedDocumentField at 0x7ff468675ed0>),
             ('created_at',
              <fiftyone.core.fields.DateTimeField at 0x7ff468675b10>),
             ('last_modified_at',
              <fiftyone.core.fields.DateTimeField at 0x7ff468675e90>),
             ('split', <fiftyone.core.fields.StringField at 0x7ff46c763b50>),
             ('bonid', <fiftyone.core.fields.IntField at 0x7ff46f8dd750>),
             ('bon_id', <fiftyone.core.fields.ListField at 0x7ff46bb6b190>),
             ('article_number',
              <fiftyone.core.fields.ListField at 0x7ff46a8aac10>),
             ('ingredient_name',
              <fiftyone.core.fields.ListField at 0x7ff469d77b10>),
             ('

## Similarity index

The similarity index allows to use an image to query its most similar samples.

In [ ]:
import fiftyone.zoo as foz
import fiftyone.brain as fob

# Whole image similarity
fob.compute_similarity(
    fiftyone_dataset,
    model=embedding_model,
    embeddings="dinov2-image-embeddings",
    brain_key='dinov2_large_reg_image_similarity_index',
    batch_size=32,
)



Computing embeddings...


INFO:fiftyone.brain.internal.core.utils:Computing embeddings...


Model does not support batching


 100% |█████████████████| 375/375 [10.8m elapsed, 0s remaining, 0.3 samples/s]    


INFO:eta.core.utils: 100% |█████████████████| 375/375 [10.8m elapsed, 0s remaining, 0.3 samples/s]    


In [ ]:
# Compute similarity for the YOLO-E segmented patches
fob.compute_similarity(
    fiftyone_dataset,
    model=embedding_model,
    patches_field="yoloe_segmentation",
    embeddings_field="segment_embeddings",  # The embeddings you computed earlier
    brain_key="segment_similarity"
)


## UMAP dimensionality reduction and visualization

We use [UMAP (Uniform Manifold Approximation and Projection)](https://umap-learn.readthedocs.io/en/latest/) to reduce the dimensionality of our image and patch embeddings. This allows us to visualize these high-dimensional data points in a 2D or 3D space, making it easier to explore the dataset's structure, identify clusters of similar images or objects, and spot outliers.

In [ ]:
visualization = fob.compute_visualization(fiftyone_dataset,
                                          method='umap',
                                          embeddings='dinov2-vitl4-embeddings',
                                          brain_key="umap_visualization")

INFO:fiftyone.core.models:Downloading model from 'https://download.pytorch.org/models/mobilenet_v2-b0353104.pth'...


 100% |████|  108.4Mb/108.4Mb [42.3ms elapsed, 0s remaining, 2.5Gb/s]       


INFO:eta.core.utils: 100% |████|  108.4Mb/108.4Mb [42.3ms elapsed, 0s remaining, 2.5Gb/s]       


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 209MB/s]


...

Generating visualization...


INFO:fiftyone.brain.visualization:Generating visualization...


UMAP( verbose=True)
Fri Jul 11 15:55:48 2025 Construct fuzzy simplicial set
Fri Jul 11 15:55:49 2025 Finding Nearest Neighbors
Fri Jul 11 15:55:51 2025 Finished Nearest Neighbor Search
Fri Jul 11 15:55:54 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Fri Jul 11 15:55:57 2025 Finished embedding


In [ ]:
fiftyone_dataset.save()

## Launch FiftyOne Session

A FiftyOne session is an interactive environment that allows you to visualize, explore, and curate your datasets. It provides a graphical user interface (GUI) where you can view samples, view the labels (segmentations) that we have produced, filter data, analyze metadata, compute aggregations, add tags, and interact with your dataset to gain insights and improve data quality.

In [ ]:
session = fo.launch_app(fiftyone_dataset, auto=False)
print(session.url)

Session launched. Run `session.show()` to open the App in a cell output.


INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.



Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.7.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



INFO:fiftyone.core.session.session:
Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.7.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



https://5151-gpu-a100-s-3eo7xrdu3jkl5-c.asia-southeast1-0.prod.colab.dev?polling=true


## Push FiftyOne dataset to HuggingFace

Share the dataset. [Upload to the HuggingFace Hub](https://docs.voxel51.com/integrations/huggingface.html#pushing-datasets-to-the-hub).



When you run this code, a few things happen:

* The dataset and its media files are exported to a temporary directory and uploaded to the specified Hugging Face repo.

* A fiftyone.yml config file for the dataset is generated and uploaded to the repo, which contains all of the necessary information so that the dataset can be loaded with `load_from_hub()`.

* A Hugging Face Dataset Card for the dataset is auto-generated, providing tags, metadata, license info, and a code snippet illustrating how to load the dataset from the hub.

* The dataset will be available on the HuggingFace Hub at the following URL: https://huggingface.co/datasets/your-username-or-org-name/food-waste-dataset

In [ ]:
from fiftyone.utils.huggingface import push_to_hub

push_to_hub(fiftyone_dataset,
            "food-waste-dataset",
            license="mit",
            exist_ok=True, # exist_ok allows you to update the remote dataset
            chunk_size=100)



Directory '/tmp/tmps3z7yln5' already exists; export will be merged with existing files


Exporting samples...


INFO:fiftyone.utils.data.exporters:Exporting samples...


 100% |████████████████████| 375/375 [391.5ms elapsed, 0s remaining, 957.8 docs/s]      


INFO:eta.core.utils: 100% |████████████████████| 375/375 [391.5ms elapsed, 0s remaining, 957.8 docs/s]      
Uploading media files in 8 batches of size 100: 100%|██████████| 8/8 [00:07<00:00,  1.10it/s]


## Summary and take-aways

In this notebook, we demonstrated a workflow for curating a computer vision dataset using HuggingFace and FiftyOne. We started by loading the Food Waste dataset from HuggingFace and improved its usability by translating German feature and ingredient names to English.

We then moved the dataset into FiftyOne, which allowed us to wrap together our workflow to:

- **Compute metadata and aggregations for samples**  providing insights into the dataset
- **Generate segmentations and bounding boxes** using a YOLO-E model to identify ingredients on plates.
- **Compute patch embeddings** from the segmented regions using DINOv2, enabling localized similarity search on objects.
- **Compute whole image similarity** using DINOv2 embeddings for overall image comparisons.
- **Visualize the dataset** using UMAP dimensionality reduction to explore the data distribution and identify potential clusters or outliers.

Finally, we showed how to **push the curated FiftyOne dataset to the HuggingFace Hub**, making it shareable and accessible for others.

This tutorial highlights how FiftyOne can enhance the workflow for preparing and understanding computer vision datasets.

## References

* [FiftyOne documentation](https://docs.voxel51.com/)
* [UMAP](https://umap-learn.readthedocs.io/)
* [DINOv2](https://learnopencv.com/dinov2-self-supervised-vision-transformer/)
* [YOLO-E](https://docs.ultralytics.com/models/yoloe/)